In [ ]:

!pip install -q openai

import json
import functools
from openai import OpenAI, RateLimitError, APIConnectionError, AuthenticationError

# 1. Groq API Configuration
MODEL = "openai/gpt-oss-20b"

client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

# 2. Bakery Menu
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

# 3. System Instructions
SYSTEM = f"""
ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Menu with prices in rupees: {json.dumps(MENU)}
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Never promise a delivery time.
If information is not available, say "I'll check with the owner."
Remember the customer's name and order during the chat.

FORMAT:
Under 50 words.
Friendly English or Telugu matching the customer.
"""

# 4. Logging Decorator
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

# 5. Price Calculation Function
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for the quantity of an item."""
    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not available in the menu.")

    if qty <= 0:
        raise ValueError("Quantity must be greater than zero.")

    return MENU[item] * qty

# 6. Chat History
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

# 7. Chat Function
def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except AuthenticationError:
        reply = "Invalid API key. Please check your Groq API key."

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."

    except Exception as e:
        reply = f"Something went wrong: {str(e)}"

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

# 8. Test Price Calculation
print("Total for 6 cupcakes:", get_price("cupcake", 6))

# 9. Start Chatbot
print("\n================================")
print(" Welcome to Sweet Crumbs Bakery!")
print(" Type 'bye' to exit.")
print("================================\n")

while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break

    print("Bot:", chat(text))


[log] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480

 Welcome to Sweet Crumbs Bakery!
 Type 'bye' to exit.

You: hii
Bot: Hi! I'm from Sweet Crumbs Home Bakery. May I know your name and what you'd like to order? Please note we need 24‑hour notice. All cakes are eggless.
You: What is the price of plum cake
Bot: The plum cake is ₹500. Let me know your name and if you’d like to place an order (24‑hour notice required).
You: What is the price of cupcakes
Bot: Cupcakes are ₹80 each. Let me know your name and if you’d like to order (24‑hour notice required). All cakes are eggless.
You: Do you have eggless cakes
Bot: Yes, all our cakes—including plum cake, chocolate cake, and cupcakes—are available in eggless versions. Let me know your name and what you’d like to order (24‑hour notice required).
You: Can I get my order tomorrow
Bot: I’m sorry, we need a 24‑hour notice for orders. Could you share your name and what you’d like to order?
You: Bye
Bot: Thank you for visiting Sweet Crum

In [ ]:

!pip install -q openai

import json
import functools
from getpass import getpass
from openai import OpenAI, RateLimitError, APIConnectionError, AuthenticationError

# 1. Groq API Configuration
MODEL = "openai/gpt-oss-20b"

api_key = getpass("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 ")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

# 2. Restaurant Menu
MENU = {
    "burger": 120,
    "pizza": 250,
    "pasta": 180,
    "sandwich": 100,
    "french fries": 90
}

# 3. System Instructions
SYSTEM = f"""
ROLE:
You are a friendly assistant for Foodie Hub Restaurant.

FACTS:
Menu: {json.dumps(MENU)}
Delivery takes 45 minutes.
Vegetarian options are available.

RULES:
Never invent menu prices.
Remember customer details.
If information is unavailable, say
"I'll check with the restaurant."

FORMAT:
Friendly responses under 50 words.
"""

# 4. Logging Decorator
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print("[LOG]", result)
        return result
    return wrapper

# 5. Bill Calculation Function
@log_call
def calculate_bill(item, qty=1):

    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not available!")

    if qty <= 0:
        raise ValueError("Quantity must be greater than zero!")

    total = MENU[item] * qty

    return total

# 6. Chat History
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

# 7. Chat Function
def chat(user_text):

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except AuthenticationError:
        reply = "Invalid API key. Please check your Groq API key."

    except RateLimitError:
        reply = "Please try again later."

    except APIConnectionError:
        reply = "Server connection failed."

    except Exception as e:
        reply = f"Error: {e}"

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

# 8. Test Bill Calculation
print("\nBill:", calculate_bill("pizza", 2))

# 9. Start Chatbot
print("\n==============================")
print(" Welcome to Foodie Hub!")
print(" Type 'bye' to exit.")
print("==============================\n")

while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting!")
        break

    print("Bot:", chat(text))



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 ··········
[LOG] 500

Bill: 500

 Welcome to Foodie Hub!
 Type 'bye' to exit.

You: hii
Bot: Invalid API key. Please check your Groq API key.
You: Bye
Bot: Thank you for visiting!
